In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output
from scipy.spatial import cKDTree
from typing import Tuple, Optional, Dict
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')
from modelnet40_dataset import ModelNet40Dataset
%load_ext autotime

time: 98.9 μs (started: 2026-04-22 18:40:36 +07:00)


In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# 2-D  Plotly interactive 3-D scatter (toggle via dropdown)
# ─────────────────────────────────────────────────────────────────────────────

def _scalar_to_color(values: np.ndarray, cmap_name: str = "jet", pct: float = 2.0):
    """Clip to [pct, 100-pct] percentile, return hex colors list."""
    lo = np.percentile(values, pct)
    hi = np.percentile(values, 100 - pct)
    normed = np.clip((values - lo) / (hi - lo + 1e-12), 0, 1)
    cmap   = plt.get_cmap(cmap_name)
    rgba   = cmap(normed)  # (N, 4)
    hex_colors = [
        "#{:02x}{:02x}{:02x}".format(int(r*255), int(g*255), int(b*255))
        for r, g, b, _ in rgba
    ]
    return hex_colors, lo, hi


def build_colorbar_trace(values: np.ndarray, cmap_name: str, lo: float, hi: float):
    """Invisible scatter used only for showing the colorbar."""
    return go.Scatter3d(
        x=[None], y=[None], z=[None],
        mode="markers",
        marker=dict(
            colorscale=cmap_name,
            cmin=lo, cmax=hi,
            color=[lo, hi],
            showscale=True,
            colorbar=dict(thickness=15, len=0.7, title=dict(side="right")),
        ),
        showlegend=False,
        hoverinfo="none",
    )


def interactive_curvature_plot(
    points: np.ndarray,
    fields: Dict[str, np.ndarray],
    cmap_name: str = "jet",
    point_size: int = 3,
):
    """
    Build a Plotly figure with one Scatter3d trace per curvature field.
    A dropdown menu (updatemenus) toggles visibility so only one field
    is shown at a time — no Python callbacks needed, works fully offline.
    """
    x, y, z = points[:, 0], points[:, 1], points[:, 2]
    field_names = list(fields.keys())
    n_fields    = len(field_names)

    traces = []
    for name, vals in fields.items():
        hex_colors, lo, hi = _scalar_to_color(vals, cmap_name)
        trace = go.Scatter3d(
            x=x, y=y, z=z,
            mode="markers",
            marker=dict(
                size=point_size,
                color=vals,
                colorscale=cmap_name,
                cmin=lo, cmax=hi,
                showscale=True,
                colorbar=dict(
                    thickness=15, len=0.6,
                    title=dict(text=name, side="right", font=dict(size=11)),
                ),
            ),
            text=[f"{name}: {v:.4f}" for v in vals],
            hoverinfo="text",
            name=name,
            visible=(name == field_names[0]),   # only first shown initially
        )
        traces.append(trace)

    # Dropdown buttons: each button shows exactly one trace
    buttons = []
    for i, name in enumerate(field_names):
        visibility = [j == i for j in range(n_fields)]
        buttons.append(
            dict(
                label=name,
                method="update",
                args=[
                    {"visible": visibility},
                    {"title": f"<b>{name}</b>"},
                ],
            )
        )

    layout = go.Layout(
        title=f"<b>{field_names[0]}</b>",
        scene=dict(
            xaxis=dict(showbackground=False, title="X"),
            yaxis=dict(showbackground=False, title="Y"),
            zaxis=dict(showbackground=False, title="Z"),
            aspectmode="data",
        ),
        updatemenus=[
            dict(
                type="dropdown",
                direction="down",
                showactive=True,
                x=0.01, xanchor="left",
                y=0.99, yanchor="top",
                buttons=buttons,
                bgcolor="white",
                bordercolor="#aaa",
                font=dict(size=12),
            )
        ],
        margin=dict(l=0, r=0, t=40, b=0),
        height=700,
        paper_bgcolor="#f8f8f8",
    )

    fig = go.Figure(data=traces, layout=layout)
    return fig


print("Visualisation functions defined.")

Visualisation functions defined.
time: 1.16 ms (started: 2026-04-22 18:40:38 +07:00)


In [3]:
ds = ModelNet40Dataset(root="out/ModelNet40", csv_path="out/metadata_modelnet40.csv",
                       split=None, compute_feats=True,n_points=1024)

time: 25.2 ms (started: 2026-04-22 18:40:40 +07:00)


In [4]:
ds

ModelNet40Dataset(split='None', n_samples=12311, n_points=1024, classes=40, normalize=True, augment=False, compute_feats=True)

time: 3.08 ms (started: 2026-04-22 18:40:41 +07:00)


In [5]:
ds.preprocess_all(cache_dir="cache_modelnet40")

[ModelNet40Dataset] Preprocessing 12311 'all' samples → cache_modelnet40


preprocess(all): 100%|██████████████████████████████████████████████████████████| 12311/12311 [00:11<00:00, 1082.60it/s]

[ModelNet40Dataset] Done. 12311 samples ready.
time: 9.02 s (started: 2026-04-22 18:40:42 +07:00)


In [6]:
ds.statistic_feats()

[ModelNet40Dataset] Feature statistics (max / mean / min):
      k1(kmin) : max=+26.8076  mean=-1.7165  min=-412.8472
      k2(kmax) : max=+539.4042  mean=+1.8926  min=-185.3272
     mean_curv : max=+131.9062  mean=+0.0881  min=-293.5854
    gauss_curv : max=+74472.5625  mean=-1.4051  min=-207626.7812
    curvedness : max=+468.5719  mean=+2.6259  min=+0.0000
     shape_idx : max=+1.0000  mean=+0.0289  min=-0.9998


[[26.807600021362305, -1.7164506912231445, -412.84722900390625],
 [539.4042358398438, 1.8926082849502563, -185.32723999023438],
 [131.90621948242188, 0.0880790427327156, -293.5854187011719],
 [74472.5625, -1.4050803184509277, -207626.78125],
 [468.5719299316406, 2.625868320465088, 0.0],
 [0.9999924898147583, 0.028942175209522247, -0.9998112320899963]]

time: 276 ms (started: 2026-04-22 18:40:55 +07:00)


In [7]:
ds._export_to_image(channel=[1,5,4], n_pix=64*2, out_dir="./train_modelnet40_full")


Exporting images: 12606464/12606464  [xbox_0103  pid=1023]023]
[ModelNet40Dataset] Saved 12606464 images → ./train_modelnet40_full
time: 1h 3min 1s (started: 2026-04-22 18:41:00 +07:00)


In [7]:
ds._export_to_image(channel=[1,5,4], n_pix=64*2, out_dir="./train_modelnet10_local_subset",local_lim=True)

Exporting images: 614400/614400  [toilet_0424  pid=1023]1023]
[ModelNet40Dataset] Saved 614400 images → ./train_modelnet10_local_subset


In [9]:
sample = ds[100]
print(sample)

{'points': tensor([[ 0.0929, -0.3159, -0.0497],
        [ 0.0815, -0.2722,  0.3841],
        [-0.0659,  0.1182,  0.4091],
        ...,
        [ 0.0803, -0.3144,  0.6163],
        [ 0.3189, -0.2822, -0.4339],
        [ 0.0755, -0.1227, -0.4339]]), 'normals': tensor([[ 0.0000,  0.0398, -0.9992],
        [ 1.0000,  0.0000,  0.0000],
        [-1.0000,  0.0000,  0.0000],
        ...,
        [ 0.0000,  0.0000,  1.0000],
        [ 0.0000,  0.0000,  1.0000],
        [ 0.0000,  0.0000,  1.0000]]), 'feats': tensor([[-2.5145, -1.2083, -1.8614,  3.0383,  1.9726, -0.7852],
        [-0.9877,  1.6237,  0.3180, -1.6037,  1.3439,  0.1521],
        [ 1.1059,  1.2973,  1.2016,  1.4347,  1.2054,  0.9494],
        ...,
        [-2.9143,  0.8917, -1.0113, -2.5988,  2.1550, -0.3110],
        [-0.2340,  0.7295,  0.2478, -0.1707,  0.5417,  0.3024],
        [-0.5228,  2.0593,  0.7683, -1.0765,  1.5023,  0.3417]]), 'radius': tensor([0.2497, 0.2877, 0.3323,  ..., 0.2372, 0.2363, 0.1196]), 'label': tensor(5), 'n

In [10]:

fields = {
        "k1 (kmin)"          : sample['feats'][:,0],
        "k2 (kmax)"          : sample['feats'][:,1],
        "mean curvature"     : sample['feats'][:,2],
        "gaussian curvature" : sample['feats'][:,3],
        "curvedness"         : sample['feats'][:,4],
        "shape index"        : sample['feats'][:,5],
        "adaptive radius"    : sample['radius'],
    }


In [13]:
fig = interactive_curvature_plot(
    sample['points'], fields,
    cmap_name  = 'jet'  ,
    point_size = 3,
)
fig.show()

In [8]:

import zipfile
with zipfile.ZipFile("volume_constrained-everyday_compressed.zip", 'r') as zip_ref:
    zip_ref.extractall('out/')

time: 8.35 s (started: 2026-04-22 22:31:15 +07:00)
